# İstanbul Baraj Doluluk Oranı Tahmini (30 gün sonrası)
Veri: İBB Açık Veri Portalı (günlük baraj doluluk + baraj bazında yağış ve İstanbul günlük tüketim), eğitim 2011-2022, test 2023.
Görev: Bugünkü bilgilerle **30 gün sonraki toplam baraj doluluk oranını (%)** tahmin etmek.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

VERI = "../data/raw/"   # Excel dosyalarının bulunduğu klasör
UFUK = 30     # kaç gün sonrasını tahmin ediyoruz

## 1) Veriyi oku ve temizle
Doluluk dosyasında 1.4.2023 - 8.8.2023 arası 130 satır **metin tarih (G.A.YYYY)** ve **yüzde (0-100)** biçiminde girilmiş;
diğer satırlar 0-1 arası oran. Bunları düzeltiyoruz.

In [ ]:
dol = pd.read_excel(VERI + "istanbul-dams-daily-occupancy-rates.xlsx")
metin = dol["Tarih"].map(lambda x: isinstance(x, str))
dol.loc[metin, "Tarih"] = pd.to_datetime(dol.loc[metin, "Tarih"], format="%d.%m.%Y")
dol["Tarih"] = pd.to_datetime(dol["Tarih"])
barajlar = [c for c in dol.columns if c != "Tarih"]
dol.loc[metin, barajlar] = dol.loc[metin, barajlar] / 100
dol = dol.set_index("Tarih").sort_index()
print("düzeltilen satır:", metin.sum(), "| tekrar eden tarih:", dol.index.duplicated().sum(), "| en büyük değer:", dol.values.max().round(3))

# Toplam doluluk = kapasiteye göre ağırlıklı ortalama (İSKİ aktif kapasiteleri, milyon m3)
kapasite = {"Ömerli": 235.371, "Darlık": 107.500, "Elmalı": 9.600, "Terkos": 162.241, "Alibey": 34.143,
            "Büyükçekmece": 148.943, "Sazlıdere": 88.730, "Kazandere": 17.424, "Pabuçdere": 58.500, "Istrancalar": 6.231}
k = pd.Series(kapasite)
dol["toplam"] = (dol[k.index] * k).sum(axis=1) / k.sum() * 100   # yüzde

yt = pd.read_excel(VERI + "istanbul-barajlarnda-ya-ve-gunluk-tuketim-verileri.xlsx")
yt = yt.drop(columns=[c for c in yt.columns if str(c).startswith("Unnamed")]).set_index("Tarih").sort_index()
yt = yt.rename(columns={"İstanbul günlük tüketim(m³/gün)": "tuketim"})
yagis_kol = [c for c in yt.columns if c != "tuketim"]
yt["yagis"] = yt[yagis_kol].mean(axis=1)      # barajların ortalama günlük yağışı (mm)

df = dol[["toplam"]].join(yt[["yagis", "tuketim"]], how="inner")
df = df.asfreq("D").interpolate(limit=3)
df = df["2011-01-01":"2023-12-31"]            # eğitim 2011-2022, test 2023
print(df.index.min().date(), "-", df.index.max().date(), df.shape)
df.describe()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
df["toplam"].plot(ax=ax[0], title="Toplam baraj doluluğu (%)")
df["yagis"].rolling(30).sum().plot(ax=ax[1], title="Son 30 gün yağış (mm)")
plt.tight_layout(); plt.show()

## 2) Öznitelikler (yalnızca bugüne kadar bilinenler) ve hedef

In [ ]:
X = pd.DataFrame(index=df.index)
X["doluluk"] = df["toplam"]
for g in [7, 30, 90]:
    X[f"degisim_{g}"] = df["toplam"] - df["toplam"].shift(g)
    X[f"yagis_{g}"] = df["yagis"].rolling(g).sum()
    X[f"tuketim_{g}"] = df["tuketim"].rolling(g).mean() / 1e6
X["ay_sin"] = np.sin(2 * np.pi * df.index.dayofyear / 365.25)
X["ay_cos"] = np.cos(2 * np.pi * df.index.dayofyear / 365.25)

hedef = df["toplam"].shift(-UFUK) - df["toplam"]   # 30 günlük değişimi öğreniyoruz
veri = X.assign(hedef=hedef).dropna()

# Zamana göre bölme (karıştırmak yok!)
# Eğitim 2011-2022: hedefi (30 gün sonrası) de 2022 içinde kalan satırlar, yani 1 Aralık 2022'ye kadar.
# Doğrulama: model ayarı için eğitim döneminin son iki yılı (2021-2022).
# Test: 2023 (hedefi de 2023 içinde kalan satırlar, yani 1 Aralık 2023'e kadar).
son_egitim = pd.Timestamp("2022-12-31") - pd.Timedelta(days=UFUK)
egitim_tum = veri[:son_egitim]
egitim = egitim_tum[:"2020-12-31"]; dogrulama = egitim_tum["2021-01-01":]
test = veri["2023-01-01":pd.Timestamp("2023-12-31") - pd.Timedelta(days=UFUK)]
oz = list(X.columns)
print(len(egitim), len(dogrulama), len(test))

## 3) Modeller ve başarı
Tahmin edilen değişim bugünkü doluluğa eklenip 30 gün sonraki doluluk elde edilir; metrikler doluluk (%) üzerindedir.

In [ ]:
def olc(ad, tahmin_degisim, parca):
    gercek = parca["doluluk"] + parca["hedef"]
    tahmin = parca["doluluk"] + tahmin_degisim
    return {"model": ad, "MAE": mean_absolute_error(gercek, tahmin),
            "RMSE": mean_squared_error(gercek, tahmin) ** 0.5, "R2": r2_score(gercek, tahmin)}

# Taban çizgisi 1: doluluk değişmez (persistence)
# Taban çizgisi 2: mevsimsel ortalama değişim (eğitim yıllarında aynı gündeki ortalama 30 günlük değişim)
mevsim = egitim_tum.groupby(egitim_tum.index.dayofyear)["hedef"].mean()
modeller = {
    "Doğrusal Regresyon": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0),
}
sonuc = []
for parca_ad, parca in [("doğrulama", dogrulama), ("test", test)]:
    sonuc.append({**olc("Taban: değişmez", np.zeros(len(parca)), parca), "küme": parca_ad})
    sonuc.append({**olc("Taban: mevsimsel", parca.index.dayofyear.map(mevsim).fillna(0).values, parca), "küme": parca_ad})
for ad, m in modeller.items():
    m.fit(egitim[oz], egitim["hedef"])                      # 2011-2020 ile eğit, 2021-2022'de ölç
    sonuc.append({**olc(ad, m.predict(dogrulama[oz]), dogrulama), "küme": "doğrulama"})
    m.fit(egitim_tum[oz], egitim_tum["hedef"])              # son model: 2011-2022'nin tamamıyla eğit
    sonuc.append({**olc(ad, m.predict(test[oz]), test), "küme": "test"})
tablo = pd.DataFrame(sonuc).pivot(index="model", columns="küme", values=["MAE", "RMSE", "R2"]).round(2)
tablo

In [ ]:
m = modeller["Gradient Boosting"]
plt.figure(figsize=(12, 4))
(test["doluluk"] + test["hedef"]).plot(label="gerçek")
pd.Series(test["doluluk"].values + m.predict(test[oz]), index=test.index).plot(label="tahmin (GB)")
plt.title(f"Test dönemi: {UFUK} gün sonraki toplam doluluk (%)"); plt.legend(); plt.show()

rf = modeller["Random Forest"]
pd.Series(rf.feature_importances_, index=oz).sort_values().plot.barh(figsize=(6, 5), title="Öznitelik önemi (RF)"); plt.show()